# Lecture 9 Activity – How does the United States compare in a 26-country road-deaths table?

QSS 20 · Modern Statistical Computing · Fall 2026

## The question

In lecture, US traffic deaths rose by 31 percent from 2013 to 2021, while New
York City's held level. A rise in one country can be part of a trend in many
countries, or a pattern of that country alone. Compare the United States
with the countries selected for the article's table.

Open [Wikipedia's Vision Zero page](https://en.wikipedia.org/wiki/Vision_Zero)
and find the table captioned "Road fatalities in 2013." Inspect its header
and one country row before running the starter code. Sweden's parliament
approved Vision Zero in 1997. The article compares 26 countries in 2013,
with deaths per million residents and per 100 billion vehicle-kilometers.
The first measure describes risk to residents; the second accounts for how
much people drive. The countries in this table are the comparison group.

> **Where does the United States rank among these 26 countries in 2013 on
> each measure?**

### How to work

Work in pairs for 35 minutes. Submit a 26-row country table, a checked US
comparison, a chart, and a short interpretation. Show the diagnostic output
that supports each claim. If you run out of time, finish Task 5 after class.

- This activity needs `beautifulsoup4`. If `import bs4` fails, run
  `conda activate qss20` and then `conda install beautifulsoup4` in a terminal.
- The row unit you build is **one country**.
- The page was saved on 2026-10-08 as `data/raw/wikipedia/vision-zero.html`.
  The live page may change. All code uses this saved copy and sends no requests.
- Print one complete row before you loop, and check your table against
  numbers the page reports.
- In the starter cells, replace each `...` with code.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from bs4 import BeautifulSoup

pd.set_option('display.max_rows', 30)

# Sharp figures on high-resolution screens, from lecture 6.
%config InlineBackend.figure_format = 'retina'

RAW = Path('../../data/raw')
DERIVED = Path('../../data/derived')   # tables saved in lecture 9
html = (RAW / 'wikipedia' / 'vision-zero.html').read_text(encoding='utf-8')
soup = BeautifulSoup(html, 'html.parser')

# Check: the page holds four tables of class wikitable.
len(soup.select('table.wikitable'))

## Task 1 · Find the right table

First run `soup.select_one('table.wikitable')` and print its caption: is it
the table you need? Then print the caption of every `wikitable` on the page,
and select the one whose caption contains `'Road fatalities in 2013'` as
`table`.

How many rows does `table` hold, and how many of them are countries?



<details><summary>Hint</summary>

`t.find('caption')` returns a table's caption element, or `None` if it has
none; `.get_text(' ', strip=True)` reads it. A list comprehension with an `if`
keeps the tables whose caption text contains the phrase.

</details>

In [ ]:
first = soup.select_one('table.wikitable')
print(first.find('caption').get_text(' ', strip=True))

for t in soup.select('table.wikitable'):
    print(...)

table = [t for t in soup.select('table.wikitable')
         if ... in t.find('caption').get_text()][0]
len(table.find_all('tr'))

*Your answer:*

## Task 2 · One row per country

Print the cells of the header row and of one country row. Then remove the
footnote marks and build a DataFrame `countries` with six columns: `country`,
`killed_1980`, `killed_2013`, `pct_of_1980`, `per_million`, and
`per_100bn_km`. Convert every column except `country` to numbers.

Which tag holds each country's name? Which column has missing values, and how
many? Explain why a missing distance rate must not become zero.



<details><summary>Hint</summary>

Look at the HTML of one row with `print(row.prettify())`. If the country's
name is not a `td`, search for `['th', 'td']`. Remove footnotes with
`for s in table.select('sup.reference'): s.decompose()`. For numbers,
`pd.to_numeric(s.str.replace(',', ''), errors='coerce')`.

</details>

In [ ]:
rows = table.find_all('tr')
print([c.get_text(' ', strip=True) for c in rows[0].find_all(['th', 'td'])])
print(rows[1].prettify())

for s in table.select(...):
    s.decompose()

columns = ['country', 'killed_1980', 'killed_2013', 'pct_of_1980',
           'per_million', 'per_100bn_km']
records = []
for row in rows[1:]:
    cells = [c.get_text(' ', strip=True) for c in row.find_all(...)]
    records.append(dict(zip(columns, cells)))
countries = pd.DataFrame(records)

for col in columns[1:]:
    countries[col] = ...
len(countries), countries['country'].is_unique, countries.isna().sum()

*Your answer:*

## Task 3 · Check the table against itself and against lecture

`pct_of_1980` should equal `killed_2013 / killed_1980 * 100`. Compute it
yourself and print the countries where your value and the page's value differ
by more than 0.5.

Then compare the US count for 2013 with the national table from lecture,
`data/derived/us_traffic_deaths.csv`. Do the two pages agree?



<details><summary>Hint</summary>

Add your own column with `.assign`, then keep the rows where the absolute
difference, `(a - b).abs()`, is above 0.5. Read the CSV with `pd.read_csv` and
select the row where `year == 2013`. `.loc[condition, 'column']` returns a
Series even when one row matches; `.item()` takes out its single value.

</details>

In [ ]:
checked = countries.assign(own_pct=...)
gap = (checked['own_pct'] - checked['pct_of_1980']).abs()
checked.loc[gap > 0.5, ['country', 'pct_of_1980', 'own_pct']].round(1)

us_year = pd.read_csv(DERIVED / 'us_traffic_deaths.csv')
(countries.loc[countries['country'] == 'United States', 'killed_2013'].item(),
 ...)

*Your answer:*

## Task 4 · ❓ Where does the United States rank?

Draw `per_million` for all 26 countries as a horizontal bar chart, sorted,
with the United States in a different color. Then report the US rank on three
measures: `per_million`, `pct_of_1980` (higher means deaths fell less), and
`per_100bn_km` (among the countries that report it).

Is the United States an outlier on every measure?



<details><summary>Hint</summary>

`df.sort_values('per_million')` puts the largest value last, which is the top
bar of a `barh` chart. Colors can be a list:
`['#b35900' if c == 'United States' else '#00693e' for c in df['country']]`.
`s.rank(ascending=False)` gives 1 to the largest value and skips `NaN`.

</details>

In [ ]:
ordered = countries.sort_values('per_million')
colors = ...

fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(ordered['country'], ordered['per_million'], color=colors)
ax.set(xlabel='Road deaths per million residents, 2013', title=...);

ranks = countries.assign(
    rank_per_million=countries['per_million'].rank(ascending=False),
    rank_pct_of_1980=...,
    rank_per_km=...)
ranks.loc[ranks['country'] == 'United States', ...]

*Your answer:*

## Task 5 · What the table cannot tell you

Write three or four sentences. What does it mean that the US ranks first per
resident but fifth per distance driven? Name two limits of this table as
evidence about road safety policy.

*Your answer:*